## 1、如何创建agent
通过调用langchain.agents.create_agent来构建一个agent，传入agent所能使用工具，以及agent的大脑：LLM

In [48]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
import dotenv
from langgraph.checkpoint.memory import InMemorySaver


dotenv.load_dotenv()
llm = ChatOpenAI(
    model="gpt-4o-mini"
)
tavily_search_tool = TavilySearch(max_results = 5) # 返回的搜索结果的最大数量
tools = [tavily_search_tool]
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是一个智能助手，能够选择合适的工具帮助用户解决问题"
)
llm.invoke("你好")

AIMessage(content='你好！有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaILF8yx61TLxhHWa80uvYe55dvDw', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--86058e2c-cc49-4988-a04a-f2cabc5ea5cf-0', usage_metadata={'input_tokens': 8, 'output_tokens': 10, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [47]:
from langgraph.pregel._read import PregelNode

# agent
# print(type(agent)) # 类型为CompiledStateGraph: 带有状态的图
print(agent.input_schema)
from langchain_tavily import TavilySearch
type(agent.nodes['__start__'])
agent.invoke({"key1":"你好"})

<class 'langgraph.graph.state.LangGraphInput'>


{'messages': [AIMessage(content='请告诉我您需要查找的信息或者问题，我将竭诚为您提供帮助！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 1281, 'total_tokens': 1303, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaI2cvEkYucdrMc2BeUq328q1dasR', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--a21e3b0a-5af8-47a2-9056-21028ddac7f9-0', usage_metadata={'input_tokens': 1281, 'output_tokens': 22, 'total_tokens': 1303, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

## 2、使用agent

In [4]:
# 不能直接使用字符串来invoke，会报错，
agent.invoke("帮我查看下北京的天气怎么样")


InvalidUpdateError: Expected dict, got 帮我查看下北京的天气怎么样
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE

In [5]:
# 调用需要通过messages key和消息列表作为value来调用
res = agent.invoke({"messages":[{"role":"user","content":"帮我看下北京天气怎么样"}]})

In [10]:
type(res)
print(res.keys())


dict_keys(['messages'])


In [14]:
for message in res["messages"]:
    print(message.__class__,message,end="\n\n\n\n")

<class 'langchain_core.messages.human.HumanMessage'> content='帮我看下北京天气怎么样' additional_kwargs={} response_metadata={} id='09937e7d-a333-4dda-95dd-5e4b7b3e6b93'



<class 'langchain_core.messages.ai.AIMessage'> content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 1292, 'total_tokens': 1315, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaBtzytpQjkAQwOse5t6fLZnjvtHt', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--19767290-3f33-4832-bbf6-5339097bc634-0' tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Beijing weather', 'topic': 'general'}, 'id': 'call_nvJ0cadDhkjKbjrpyMhxwvlK', 'type': 'tool_call'}] usage_metadata={'inp

In [15]:
res["messages"][-1]

AIMessage(content='当前北京的天气如下：\n\n- **温度**：9.3°C (约48.7°F)\n- **天气状况**：晴天\n- **风速**：约3.6 km/h (2.2 mph)\n- **湿度**：50%\n- **可见度**：10 km\n\n预计未来几天北京的天气将稍微变凉，预计气温在12.9°C左右，并且降雨量很少，大约在5天内降水量为15.5 mm。\n\n如需更详细的天气信息或预报，可以访问 [WeatherAPI](https://www.weatherapi.com/) 或 [AccuWeather](https://www.accuweather.com/en/cn/beijing/101924/november-weather/101924)。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 167, 'prompt_tokens': 3975, 'total_tokens': 4142, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1280}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaBu2YTBRc3OA9V3MLqF0Vf8ii4Cw', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--26f9e029-db8b-4b4c-b2dd-61124ac1be91-0', usage_metadata={'input_tokens': 3975, 'outpu

## 3、使用langsmith来追踪调用链
1. 为什么使用langsmith: agent封装了整个调用工具的交互流程，不好看到调用agent整体和llm的交互有哪些，通过langsmith就可以清晰看到
2. 使用方式：在langsmith申请api_key，构造以下环境变量：
- LANGSMITH_TRACING=true
- LANGSMITH_API_KEY=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
- LANGSMITH_ENDPOINT=https://api.smith.langchain.com
- LANGSMITH_PROJECT=agent_proj。

构造好后，load环境变量，之后langchain底层就可以自动将每次调用记录到langsmith当中了

## 4、如何在agent当中引入记忆
1. 在创建agent时，传入checkpointer参数，在测试环境下，可以传入InMemorySaver()，生产环境下，使用数据来保存记忆。
2. 在invoke时，需要指定，将记忆保存到哪个“会话”当中，指定方式，传入config参数：{"configurable":{"thread_id":1}},thread_id真正指定了，将记忆存入哪个“会话当中”

In [17]:
from langgraph.checkpoint.memory import InMemorySaver
agent_with_memory = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是一个智能助手，能够选择合适的工具帮助用户解决问题",
    checkpointer=InMemorySaver()
)

In [20]:
agent_with_memory.invoke({"messages":[{"role":"user","content":"帮我看下北京天气怎么样"}]},config={"configurable":{"thread_id":1}})

{'messages': [HumanMessage(content='帮我看下北京天气怎么样', additional_kwargs={}, response_metadata={}, id='0288fc5d-caf4-4cf8-b1c7-e9eaf8d18f02'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 1292, 'total_tokens': 1319, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaCDVFupY2Qachn8Ie3olpSkUPn9y', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--197b36b8-5c0a-461f-b84d-d68ce348798e-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '北京天气', 'topic': 'news', 'time_range': 'day'}, 'id': 'call_UkXmU4m6mZGH3hATUQhWYkYE', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1292, 'output_tokens': 27, 'total_t

In [22]:
agent_with_memory.invoke({"messages":[{"role":"user","content":"我刚才问你什么了"}]},config={"configurable":{"thread_id":1}})["messages"][-1]

AIMessage(content='你刚才询问了北京的天气情况。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 5809, 'total_tokens': 5822, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 5760}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaCEScIGXcvr0ghluMx84HsOkS0nk', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--64883c87-0800-48af-aa13-90afba6ea144-0', usage_metadata={'input_tokens': 5809, 'output_tokens': 13, 'total_tokens': 5822, 'input_token_details': {'audio': 0, 'cache_read': 5760}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [23]:
# 对比原生llm，无法维护消息列表状态
llm.invoke("北京天气怎么样")
llm.invoke("我刚才问你什么了")

AIMessage(content='抱歉，我无法提供实时的天气信息。不过，你可以通过天气预报网站、应用程序或查阅当地新闻获取北京的最新天气情况。如果你有其他问题或需要更具体的信息，欢迎随时问我！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 10, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaCEwk4AbtaJxQP6W87LA0RTAl0U6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--8f96e149-9e89-4c12-9182-5e916c99cad0-0', usage_metadata={'input_tokens': 10, 'output_tokens': 49, 'total_tokens': 59, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [26]:
agent_with_memory.invoke({"messages":[{"role":"user","content":"我刚才问你什么了"}]},config={"configurable":{"thread_id":1}})["messages"][-1]

AIMessage(content='你刚才问我查看北京的天气怎么样。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 5835, 'total_tokens': 5848, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 5760}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CaCGRagQCQ5uSkpvBNgc0363QP707', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--868c4d23-17b3-41b3-9f4c-a48d12fecf13-0', usage_metadata={'input_tokens': 5835, 'output_tokens': 13, 'total_tokens': 5848, 'input_token_details': {'audio': 0, 'cache_read': 5760}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [27]:
type(agent_with_memory)

langgraph.graph.state.CompiledStateGraph